<a href="https://colab.research.google.com/github/shahdelmasry12/AI-Machine-Learning/blob/main/NLP_Projects/sentiment_finetune_HF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Phase 4 - LLM (Hugging Face)Sentiment Analysis**

## 0️⃣  Install Libraries

In [1]:
!pip install transformers datasets scikit-learn torch accelerate -q

In [2]:
!pip install huggingface_hub

In [3]:
from huggingface_hub import notebook_login

notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## 1️⃣  Imports

In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

## 2️⃣  Load Dataset

In [ ]:
# ─── Change path if needed ────────────────────────────────
CSV_PATH = 'sentiment_data.csv'   # ← ضعي الملف في نفس مجلد الـ notebook
# ─────────────────────────────────────────────────────────

df = pd.read_csv(CSV_PATH)
df = df[['Comment', 'Sentiment']].dropna()
df['Comment'] = df['Comment'].astype(str)
df['label']   = df['Sentiment'].astype(int)

ID2LABEL = {0: 'negative', 1: 'neutral', 2: 'positive'}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}

print(f'📊 Total rows : {len(df):,}')
print(f'🏷️  Distribution:')
print(df['label'].map(ID2LABEL).value_counts())
df.head(3)

## 3️⃣  Sample & Split

> **Tip:** غيري `SAMPLE_SIZE` لـ `None` لو عايزة تتدربي على كل الداتا (بطيء بدون GPU)

In [ ]:
# None = use full dataset  |  integer = use that many rows
SAMPLE_SIZE = 30000   # ← زوديها لـ None لو عندك GPU

if SAMPLE_SIZE:
    df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=42)

train_df, temp_df = train_test_split(df,      test_size=0.2,  random_state=42, stratify=df['label'])
val_df,   test_df = train_test_split(temp_df, test_size=0.5,  random_state=42, stratify=temp_df['label'])

print(f'Train : {len(train_df):,}')
print(f'Val   : {len(val_df):,}')
print(f'Test  : {len(test_df):,}')

## 4️⃣  Load Pretrained Model & Tokenizer

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
print(f'✅ Loaded: {MODEL_NAME}')

## 5️⃣  Tokenize

In [ ]:
MAX_LEN = 128   # increase to 256 if comments are long

def tokenize(batch):
    return tokenizer(batch['Comment'], truncation=True, padding='max_length', max_length=MAX_LEN)

def make_dataset(split_df):
    ds = Dataset.from_pandas(split_df[['Comment', 'label']].reset_index(drop=True))
    ds = ds.map(tokenize, batched=True)
    ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
    return ds

train_ds = make_dataset(train_df)
val_ds   = make_dataset(val_df)
test_ds  = make_dataset(test_df)
print('✅ Tokenization done!')

## 6️⃣  Fine-Tune with Trainer API

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds)}

use_gpu = torch.cuda.is_available()

training_args = TrainingArguments(
    output_dir                  = './sentiment_model',
    num_train_epochs            = 3,
    per_device_train_batch_size = 32 if use_gpu else 16,
    per_device_eval_batch_size  = 64 if use_gpu else 32,
    warmup_steps                = 100,
    weight_decay                = 0.01,
    logging_steps               = 50,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'accuracy',
    report_to                   = 'none',
    fp16                        = use_gpu,
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    compute_metrics = compute_metrics,
)

print('🚀 Starting Fine-Tuning ...')
trainer.train()

## 7️⃣  Evaluate on Test Set

In [ ]:
pred_out = trainer.predict(test_ds)
preds    = np.argmax(pred_out.predictions, axis=-1)
labels   = pred_out.label_ids

print('=' * 58)
print('📊  CLASSIFICATION REPORT')
print('=' * 58)
print(classification_report(labels, preds, target_names=['negative', 'neutral', 'positive']))

print('\n🔢 Confusion Matrix (rows=actual, cols=predicted):')
cm = confusion_matrix(labels, preds)
cm_df = pd.DataFrame(cm, index=['actual_neg','actual_neu','actual_pos'],
                         columns=['pred_neg','pred_neu','pred_pos'])
print(cm_df)

## 8️⃣  Save Fine-Tuned Model

In [ ]:
SAVE_PATH = './my_sentiment_model'
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f'✅ Model saved → {SAVE_PATH}')

## 9️⃣  Inference — Predict on New Comments

In [ ]:
clf = pipeline(
    'text-classification',
    model=SAVE_PATH,
    tokenizer=SAVE_PATH,
    device=0 if use_gpu else -1,
)

new_comments = [
    "This product is absolutely amazing, highly recommend!",
    "Very disappointed, waste of money.",
    "It's okay, does the job but nothing special.",
]

print('💬 Predictions on New Comments')
print('-' * 55)
for text, res in zip(new_comments, clf(new_comments)):
    print(f"  Label : {res['label']:<10}  Score: {res['score']:.4f}")
    print(f"  Text  : {text}")
    print('-' * 55)

## 🔟  Apply Model to Full Dataset & Export Results

In [ ]:
# Load full dataset again
full_df = pd.read_csv(CSV_PATH)
full_df = full_df[['Comment', 'Sentiment']].dropna()
full_df['Comment'] = full_df['Comment'].astype(str)

# Predict in batches (to avoid memory issues)
BATCH = 256
all_labels  = []
all_scores  = []

comments = full_df['Comment'].tolist()
print(f'🔄 Predicting on {len(comments):,} rows ...')

for i in range(0, len(comments), BATCH):
    batch = comments[i:i+BATCH]
    results = clf(batch, truncation=True, max_length=128)
    all_labels.extend([r['label']  for r in results])
    all_scores.extend([r['score']  for r in results])
    if i % 5000 == 0:
        print(f'  → {i:,}/{len(comments):,} done')

full_df['predicted_sentiment'] = all_labels
full_df['confidence_score']    = [round(s, 4) for s in all_scores]

# Export
OUT_PATH = 'sentiment_predictions.csv'
full_df.to_csv(OUT_PATH, index=False)
print(f'\n✅ Predictions saved → {OUT_PATH}')
full_df.head(5)